# Introduction to deep learning- Assignment 1


In [1]:
from tqdm.auto import tqdm,trange
import torch
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassF1Score
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torch.utils.data import Dataset
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import torch.nn.functional as F
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
print(torch.cuda.get_device_name(device)
)

Using cuda device
AMD Radeon RX 7600S


# Loading Training and Test sets from PyTorch

In [2]:
target_transform = v2.Lambda(
    lambda y: F.one_hot(torch.tensor(y), num_classes=10).float()
)
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

Trainig_data_ci = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform

)

test_data_ci = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

# MLP class using 2 hidden layers

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self, x = 0.0, y = 0.0, input = 28*28, first_hidden = 512,
                 second_hidden = 512, first_activation = nn.ReLU, second_activation = nn.ReLU):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_stack = nn.Sequential(
            nn.Linear(input, first_hidden),
            first_activation(),
            nn.Dropout(x),
            nn.Linear(first_hidden, second_hidden),
            second_activation(),
            nn.Dropout(y),
            nn.Linear(second_hidden, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_stack(x)
        return logits

# Training with MLP using Train and Validation split (with default ratio of 80/20)

In [4]:
def train_model(**kwargs):
    model         = kwargs.get("model", NeuralNetwork())
    train_dataset = kwargs.get("train_dataset", training_data)
    seed          = kwargs.get("seed", 528)
    epoches       = kwargs.get("epoches", 15)
    lr            = kwargs.get("lr", 0.003)
    optimizer_cls = kwargs.get("optimizer_cls", torch.optim.Adam)
    loss_FM       = kwargs.get("loss_fn", nn.CrossEntropyLoss())
    val_ratio     = kwargs.get("val_ratio", 0.2)
    batch_size    = kwargs.get("batch_size", 64)
    num_classes   = kwargs.get("num_classes", 10)
    average       = kwargs.get("average", "macro")
    save_path     = kwargs.get("save_path", "best_model.pt")
    dev           = kwargs.get("device", device)

    history = {
        "train_loss": [], "train_acc": [], "train_prec": [], "train_f1": [],
        "val_loss": [],   "val_acc": [],   "val_prec": [],   "val_f1": [],
    }
    best_val_acc = 0.0
    acc  = MulticlassAccuracy(num_classes=num_classes).to(dev)
    prec = MulticlassPrecision(num_classes=num_classes, average=average).to(dev)
    f1   = MulticlassF1Score(num_classes=num_classes, average=average).to(dev)
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    train_split, val_split = random_split(train_dataset, [len(train_dataset) - int(len(train_dataset) * val_ratio), int(len(train_dataset) * val_ratio)], generator=torch.Generator().manual_seed(seed))
    train_loader = DataLoader(dataset=train_split, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=val_split, batch_size=batch_size, shuffle=True)

    train_bar = tqdm(total=len(train_loader), desc="train")
    val_bar   = tqdm(total=len(val_loader),   desc="val")

    for epoch in trange(epoches, desc="Training"):

        train_loss_total = 0.0
        val_loss_total = 0.0
        model.train().to(dev)

        train_bar.reset()
        train_bar.set_description(f"epoch {epoch} train")
        for x, y in train_loader:
            x = x.to(dev)
            y = y.to(dev)
            logits = model(x)
            loss = loss_FM(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            preds = logits.argmax(dim=1)
            acc_v = acc(preds, y)
            prec_v = prec(preds, y)
            f1_v = f1(preds, y)
            train_loss_total += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc_v:.4f}",
                                  prec=f"{prec_v:.4f}", f1=f"{f1_v:.4f}")
            train_bar.update(1)

        history["train_loss"].append(train_loss_total / len(train_loader))
        history["train_acc"].append(acc.compute().item())
        history["train_prec"].append(prec.compute().item())
        history["train_f1"].append(f1.compute().item())

        tqdm.write(f"epoch {epoch} train | loss {history['train_loss'][-1]:.4f} | "
                   f"Accuracy {history['train_acc'][-1]:.4f} | Precision {history['train_prec'][-1]:.4f} | "
                   f"F1 {history['train_f1'][-1]:.4f}")
        acc.reset()
        prec.reset()
        f1.reset()

        with torch.no_grad():
            model.eval()
            val_bar.reset()
            val_bar.set_description(f"epoch {epoch} val")
            for x, y in val_loader:
                x = x.to(dev)
                y = y.to(dev)
                logits = model(x)
                loss = loss_FM(logits, y)
                preds = logits.argmax(dim=1)
                acc_v = acc(preds, y)
                prec_v = prec(preds, y)
                f1_v = f1(preds, y)
                val_loss_total += loss.item()
                val_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc_v:.4f}")
                val_bar.update(1)

        history["val_loss"].append(val_loss_total / len(val_loader))
        history["val_acc"].append(acc.compute().item())
        history["val_prec"].append(prec.compute().item())
        history["val_f1"].append(f1.compute().item())

        tqdm.write(f"epoch {epoch} val   | loss {history['val_loss'][-1]:.4f} | "
                   f"Accuracy {history['val_acc'][-1]:.4f} | Precision {history['val_prec'][-1]:.4f} | "
                   f"F1 {history['val_f1'][-1]:.4f}")
        if history["val_acc"][-1] > best_val_acc:
            best_val_acc = history["val_acc"][-1]
            torch.save(model.state_dict(), save_path)
            tqdm.write(f"new best val acc {best_val_acc:.4f}, saved")

        acc.reset()
        prec.reset()
        f1.reset()

    train_bar.close()
    val_bar.close()

    return history, best_val_acc

# Evaluation function to be used with test set

In [5]:
def evaluate_model(**kwargs):
    test_model   = kwargs.get("model", NeuralNetwork())
    test_dataset = kwargs.get("test_dataset", test_data)
    load_path    = kwargs.get("load_path", "best_model.pt")
    batch_size   = kwargs.get("batch_size", 64)
    loss_FM      = kwargs.get("loss_fn", nn.CrossEntropyLoss())
    num_classes  = kwargs.get("num_classes", 10)
    average      = kwargs.get("average", "macro")
    dev          = kwargs.get("device", device)

    acc  = MulticlassAccuracy(num_classes=num_classes).to(dev)
    prec = MulticlassPrecision(num_classes=num_classes, average=average).to(dev)
    f1   = MulticlassF1Score(num_classes=num_classes, average=average).to(dev)

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    test_model.load_state_dict(torch.load(load_path, map_location=dev))
    test_model.to(dev)
    test_model.eval()

    acc.reset(); prec.reset(); f1.reset()
    test_loss_total = 0.0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(dev), y.to(dev)
            logits = test_model(x)
            test_loss_total += loss_FM(logits, y).item()
            preds = logits.argmax(dim=1)
            acc.update(preds, y)
            prec.update(preds, y)
            f1.update(preds, y)

    print(f"test | loss {test_loss_total/len(test_loader):.4f} | "
          f"Accuracy {acc.compute():.4f} | Precision {prec.compute():.4f} | F1 {f1.compute():.4f}")

    return {
        "loss": test_loss_total / len(test_loader),
        "acc": acc.compute().item(),
        "prec": prec.compute().item(),
        "f1": f1.compute().item(),
    }

# Run 1 without dropout (FashionMNIST)

In [6]:
# Run 1: no dropout
history, best = train_model(model=NeuralNetwork(), save_path="best_model.pt")
evaluate_model(model=NeuralNetwork(), load_path="best_model.pt")



train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5109 | Accuracy 0.8124 | Precision 0.8108 | F1 0.8103
epoch 0 val   | loss 0.4130 | Accuracy 0.8460 | Precision 0.8529 | F1 0.8461
new best val acc 0.8460, saved
epoch 1 train | loss 0.3845 | Accuracy 0.8586 | Precision 0.8580 | F1 0.8579
epoch 1 val   | loss 0.4023 | Accuracy 0.8519 | Precision 0.8527 | F1 0.8507
new best val acc 0.8519, saved
epoch 2 train | loss 0.3477 | Accuracy 0.8716 | Precision 0.8711 | F1 0.8710
epoch 2 val   | loss 0.3751 | Accuracy 0.8649 | Precision 0.8709 | F1 0.8665
new best val acc 0.8649, saved
epoch 3 train | loss 0.3195 | Accuracy 0.8809 | Precision 0.8805 | F1 0.8804
epoch 3 val   | loss 0.3656 | Accuracy 0.8676 | Precision 0.8710 | F1 0.8688
new best val acc 0.8676, saved
epoch 4 train | loss 0.3044 | Accuracy 0.8865 | Precision 0.8861 | F1 0.8861
epoch 4 val   | loss 0.3459 | Accuracy 0.8717 | Precision 0.8770 | F1 0.8732
new best val acc 0.8717, saved
epoch 5 train | loss 0.2870 | Accuracy 0.8930 | Precision 0.8927 | F1 0.892

{'loss': 0.4010920221353792,
 'acc': 0.880899965763092,
 'prec': 0.8814626932144165,
 'f1': 0.8808264136314392}

# Run 2 with dropout 0.2 (FashionMNIST)

In [7]:
# Run 2: dropout 0.2 / 0.2
history_drop, best_drop = train_model(model=NeuralNetwork(x=0.2, y=0.2), save_path="best_model_drop.pt")
evaluate_model(model=NeuralNetwork(x=0.2, y=0.2), load_path="best_model_drop.pt")

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5556 | Accuracy 0.7977 | Precision 0.7956 | F1 0.7951
epoch 0 val   | loss 0.4465 | Accuracy 0.8382 | Precision 0.8424 | F1 0.8338
new best val acc 0.8382, saved
epoch 1 train | loss 0.4410 | Accuracy 0.8392 | Precision 0.8381 | F1 0.8380
epoch 1 val   | loss 0.4005 | Accuracy 0.8485 | Precision 0.8639 | F1 0.8508
new best val acc 0.8485, saved
epoch 2 train | loss 0.4127 | Accuracy 0.8497 | Precision 0.8491 | F1 0.8490
epoch 2 val   | loss 0.3634 | Accuracy 0.8676 | Precision 0.8669 | F1 0.8650
new best val acc 0.8676, saved
epoch 3 train | loss 0.3954 | Accuracy 0.8565 | Precision 0.8558 | F1 0.8557
epoch 3 val   | loss 0.3718 | Accuracy 0.8638 | Precision 0.8623 | F1 0.8609
epoch 4 train | loss 0.3866 | Accuracy 0.8596 | Precision 0.8589 | F1 0.8589
epoch 4 val   | loss 0.3675 | Accuracy 0.8678 | Precision 0.8668 | F1 0.8658
new best val acc 0.8678, saved
epoch 5 train | loss 0.3705 | Accuracy 0.8656 | Precision 0.8648 | F1 0.8648
epoch 5 val   | loss 0.3607 |

{'loss': 0.3590491635214751,
 'acc': 0.8703999519348145,
 'prec': 0.870198667049408,
 'f1': 0.8692367672920227}